# Cleaning 1.3 - Certification cleaning

This notebook does the following:
    (1) Creates counts of certification criteria with "Yes", "No", "I don't know", "Not applicable", "Unfilled"
    (2) Creates indicators for bronze/silver/gold certification: 
        - at least 1 "Yes" and 
        - for actions that are not "Not applicable" or "Unfilled", at least 70% "Yes"

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os

In [2]:
# Load data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Load list of labs with no BL checklist
labs_no_bl_checklist = pd.read_excel(config.LABS_LIST / "labs_no_BL_SPARK_checklist.xlsx")

## (1) Create counts of "Yes", "No", "I don't know", "Not applicable", "Unfilled" for each of bronze, silver, gold for BL and EL

In [4]:
# Indicators for checklist/audit at BL and EL

labs["checklist_bl"] = (~labs["labgroupid"].isin(labs_no_bl_checklist["labgroupid"])) & (labs["Treatment Status"] == "treatment")

labs["checklist_el"] = (labs["el_date_filled"] == True) & (labs["Treatment Status"] == "treatment")

print("Number of labs with BL checklist:", labs["checklist_bl"].sum())
print("Number of labs with EL checklist:", labs["checklist_el"].sum())

Number of labs with BL checklist: 64
Number of labs with EL checklist: 54


In [5]:
# Dictionary for yes "Yes" unsure "I don't know" no "No"
answer_dict = {
    "yes": "Yes",
    "no": "No",
    "unsure": "I don't know",
    "na": "Not applicable"
}

# Dictionary for no of qs for each award
qs_dict = {
    "bronze": 16,
    "silver": 18,
    "gold": 15
}

# For each survey, award, and answer in answer_dict, create a col indicating the number of qs with that answer
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        for ans in list(answer_dict) + ["empty"]:
            labs[f"{award}_{ans}_{s}"] = 0

        for q_num in range(1, qs_dict[award] + 1):
            q_col = f"{award}_q_{q_num}_{s}"
            response = labs[q_col].astype("string").str.strip()

            for ans, key in answer_dict.items():
                col_name = f"{award}_{ans}_{s}"
                labs[col_name] += response.str.lower().eq(key.lower()).fillna(False).astype(int)

            labs[f"{award}_empty_{s}"] += response.isna().astype(int)

# Check that sum of yes, no, unsure, na, and empty cols equals total number of qs for each award and survey
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        total_qs = qs_dict[award]
        sum_cols = (
            labs[f"{award}_yes_{s}"] +
            labs[f"{award}_no_{s}"] +
            labs[f"{award}_unsure_{s}"] +
            labs[f"{award}_na_{s}"] +
            labs[f"{award}_empty_{s}"]
        )
        assert (sum_cols == total_qs).all(), f"Sum of answer columns does not equal total number of questions for {award} {s}"

# For each survey and award, create a var indicating the number of applicable qs (i.e. not na or empty)
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        labs[f"{award}_app_{s}"] = labs[f"{award}_yes_{s}"] + labs[f"{award}_no_{s}"] + labs[f"{award}_unsure_{s}"]

# Print how many labs have all empty questions for each survey
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        empty_col = f"{award}_empty_{s}"
        if empty_col in labs.columns:
            num_empty = (labs[empty_col] == qs_dict[award]).sum()
            print(f"{num_empty} labs have all questions empty for {award} {s}.")

74 labs have all questions empty for bronze bl.
74 labs have all questions empty for silver bl.
74 labs have all questions empty for gold bl.
87 labs have all questions empty for bronze el.
87 labs have all questions empty for silver el.
87 labs have all questions empty for gold el.


In [6]:
# Indicators for at least one non-empty question for each survey (across awards)
for s in ["bl", "el"]:
    labs[f"any_q_filled_{s}"] = (
        (labs[f"bronze_empty_{s}"] < qs_dict["bronze"]) |
        (labs[f"silver_empty_{s}"] < qs_dict["silver"]) |
        (labs[f"gold_empty_{s}"] < qs_dict["gold"])
    )

print("Number of labs with at least one non-empty question at BL:", labs["any_q_filled_bl"].sum())
print("Number of labs with at least one non-empty question at EL:", labs["any_q_filled_el"].sum())

# Check that all labs with one non-empty question are in treatment group
assert (labs.loc[labs["any_q_filled_bl"], "Treatment Status"] == "treatment").all()
assert (labs.loc[labs["any_q_filled_el"], "Treatment Status"] == "treatment").all()

Number of labs with at least one non-empty question at BL: 64
Number of labs with at least one non-empty question at EL: 51


## (2) Certification indicators

In [ ]:
# Criteria for certification: at least 1 yes, and at least 70% of applicable qs are yes (should be missing if any_q_filled is False)
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        labs[f"{award}_cert_{s}"] = np.where(
            (labs[f"{award}_yes_{s}"] >= 1) & 
            (labs[f"{award}_yes_{s}"] / labs[f"{award}_app_{s}"] >= 0.7),
            "Certified",
            np.where(
                labs[f"any_q_filled_{s}"] == True, 
                "Not Certified", 
                pd.NA)
        )

# Print summary - for each award, how many certified, how many uncertified, percentage certified among those certified/not certified
for s in ["bl", "el"]:
    for award in ["bronze", "silver", "gold"]:
        cert_counts = labs[f"{award}_cert_{s}"].value_counts()
        cert_props = labs[f"{award}_cert_{s}"].value_counts(normalize=True).mul(100).round(2)
        print(f"{award.capitalize()} {s.upper()}:")
        print(cert_counts)
        print (f"Percentage certified among those with a certification status: {cert_props.get('Certified', 0)}%")
        print()


Bronze BL:
bronze_cert_bl
Certified        41
Not Certified    23
Name: count, dtype: int64
Percentage Certified among those with a certification status: 64.06%

Silver BL:
silver_cert_bl
Not Certified    38
Certified        26
Name: count, dtype: int64
Percentage Certified among those with a certification status: 40.62%

Gold BL:
gold_cert_bl
Certified        36
Not Certified    28
Name: count, dtype: int64
Percentage Certified among those with a certification status: 56.25%

Bronze EL:
bronze_cert_el
Certified        44
Not Certified     7
Name: count, dtype: int64
Percentage Certified among those with a certification status: 86.27%

Silver EL:
silver_cert_el
Certified        32
Not Certified    19
Name: count, dtype: int64
Percentage Certified among those with a certification status: 62.75%

Gold EL:
gold_cert_el
Certified        36
Not Certified    15
Name: count, dtype: int64
Percentage Certified among those with a certification status: 70.59%



## Save processed dataset

In [ ]:
# Save processed dataset
labs.to_csv(config.PROCESSED_DATA / "individual_processed_3.csv", index = False)